# সহজ ভাষায় Notebook Guide

এই notebook **সম্পূর্ণ self-contained** — project folder লাগে না, Docker daemon-ও লাগে না। Compose file-কে আমরা Python data হিসেবে বানাব, YAML-এ লিখব, নিজের linter দিয়ে সাধারণ ভুল ধরব, image-এর size budget হিসাব করব, আর `docker compose ps`-এর output পড়ে বলব stack *ready* কিনা। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন। দরকার: `pyyaml`, `pandas`। PyYAML না থাকলে JSON দিয়ে চলে — JSON নিজেই valid YAML, Compose দুটোই পড়ে।
2. Section 2 `docker` CLI থাকলে `docker compose config` দিয়ে file যাচাই করে — এর জন্য daemon চালু থাকতে হয় না। CLI না থাকলে cell নিজেই skip করে।
3. **Verification-এর সৎ কথা:** এই notebook যে machine-এ চালানো হয়েছে সেখানে Docker CLI ছিল কিন্তু daemon বা GPU ছিল না। তাই section 6-এর `docker compose ps` output একটা **illustrative sample**, আসল চলমান stack থেকে না — নিজের machine-এ `docker compose ps -a --format json` চালিয়ে মেলান। বাকি সব cell pure Python (বা daemon-ছাড়া CLI), output আসল।

# Containerizing the AI Stack — Lecture Notebook

1. Compose file = data: Healthcare stack (UI + API + Chroma + Ollama + model-pull init) Python dict থেকে YAML
2. `docker compose config` — Compose নিজে কী ধরে
3. Compose linter — Compose যা ধরে না: `localhost`, `latest`, healthcheck ছাড়া `service_healthy`, literal secret
4. Dockerfile linter — cache-এর ক্রম, CUDA torch, root user, shell-form `CMD`
5. Image size budget — AI image কেন ১০ GB হয়
6. `docker compose ps` পড়া — running বনাম ready
7. Single-click-এর শেষ ধাপ: readiness wait

In [1]:
import copy, json, re, shutil, subprocess, tempfile, threading, time, urllib.error, urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import pandas as pd
try:
    import yaml
except ImportError:          # JSON is valid YAML, so the notebook still works
    yaml = None

def to_yaml(spec):
    return yaml.safe_dump(spec, sort_keys=False, width=120) if yaml else json.dumps(spec, indent=2)

DOCKER = shutil.which("docker")
WORK = Path(tempfile.mkdtemp(prefix="w18_compose_"))
print("PyYAML", getattr(yaml, "__version__", "missing -> JSON fallback"), "| pandas", pd.__version__)
print("docker CLI:", "found" if DOCKER else "not found -> section 2 is skipped")

PyYAML 6.0.3 | pandas 3.1.0
docker CLI: found


## 1. Compose File = Data

Compose file আসলে একটা nested dictionary: service, তার image বা build, environment, volume, healthcheck, আর কে কার উপর নির্ভর করে। Python dict হিসেবে বানালে যাচাই করা সহজ — section 3-এর linter এই dict-ই পড়বে। নিচের stack Week 17-এর Healthcare Assistant-এর আকারে:

* `ollama` — LLM server; model থাকে `ollama_models` volume-এ। কোনো `ports` নেই — Ollama-র auth নেই।
* `ollama-init` — one-shot: model না থাকলে তবেই `llama3.1:8b` pull, তারপর exit 0।
* `chroma` — vector DB server; data `chroma_data` volume-এ।
* `api` — FastAPI; Ollama আর Chroma-কে **service name** দিয়ে ডাকে। Host-এ শুধু `127.0.0.1`-এ (debug)।
* `ui` — Streamlit; API-কে `http://api:8000`-এ ডাকে। Host-এ সবার জন্য খোলা একমাত্র port।

Image tag `.env` থেকে আসে: `${OLLAMA_TAG:?...}` — না দিলে Compose error দিয়ে থামে, চুপচাপ `latest` নেয় না।

In [2]:
MODEL = "llama3.1:8b"
PY_HEALTH = "import urllib.request as u; u.urlopen('http://localhost:{port}{path}')"

def hc(test, retries=6, start_period=None):
    h = {"test": test, "interval": "10s", "timeout": "5s", "retries": retries}
    return h | ({"start_period": start_period} if start_period else {})

STACK = {
    "name": "healthcare-assistant",          # fixed project name -> fixed volume names
    "services": {
        "ollama": {
            "image": "ollama/ollama:${OLLAMA_TAG:?pin OLLAMA_TAG in .env}",
            "volumes": ["ollama_models:/root/.ollama"],
            "healthcheck": hc(["CMD", "ollama", "list"], retries=30, start_period="20s"),   # no curl in the image
            "restart": "unless-stopped",
        },
        "ollama-init": {
            "image": "ollama/ollama:${OLLAMA_TAG:?pin OLLAMA_TAG in .env}",
            "environment": {"OLLAMA_HOST": "http://ollama:11434"},
            "entrypoint": ["sh", "-c", f"ollama show {MODEL} >/dev/null 2>&1 || ollama pull {MODEL}"],
            "depends_on": {"ollama": {"condition": "service_healthy"}},
            "restart": "no",
        },
        "chroma": {
            "image": "chromadb/chroma:${CHROMA_TAG:?pin CHROMA_TAG in .env}",
            "volumes": ["chroma_data:/data"],
            "healthcheck": hc(["CMD-SHELL", "bash -c ':> /dev/tcp/127.0.0.1/8000' || exit 1"], retries=12),
            "restart": "unless-stopped",
        },
        "api": {
            "build": {"context": ".", "dockerfile": "Dockerfile.api"},
            "environment": {"HC_OLLAMA_HOST": "http://ollama:11434", "HC_CHROMA_HOST": "chroma",
                            "HC_AUTO_INGEST": "true", "HC_API_KEY": "${HC_API_KEY:-}"},
            "ports": ["127.0.0.1:8000:8000"],
            "depends_on": {"chroma": {"condition": "service_healthy"},
                           "ollama-init": {"condition": "service_completed_successfully"}},
            "healthcheck": hc(["CMD", "python", "-c", PY_HEALTH.format(port=8000, path="/health")], start_period="120s"),
            "restart": "unless-stopped",
        },
        "ui": {
            "build": {"context": ".", "dockerfile": "Dockerfile.ui"},
            "environment": {"HC_API_URL": "http://api:8000"},
            "ports": ["8501:8501"],
            "depends_on": {"api": {"condition": "service_healthy"}},
            "healthcheck": hc(["CMD", "python", "-c", PY_HEALTH.format(port=8501, path="/_stcore/health")]),
            "restart": "unless-stopped",
        },
    },
    "volumes": {"ollama_models": {}, "chroma_data": {}},
}
print(to_yaml(STACK)[:1400], "...")

name: healthcare-assistant
services:
  ollama:
    image: ollama/ollama:${OLLAMA_TAG:?pin OLLAMA_TAG in .env}
    volumes:
    - ollama_models:/root/.ollama
    healthcheck:
      test:
      - CMD
      - ollama
      - list
      interval: 10s
      timeout: 5s
      retries: 30
      start_period: 20s
    restart: unless-stopped
  ollama-init:
    image: ollama/ollama:${OLLAMA_TAG:?pin OLLAMA_TAG in .env}
    environment:
      OLLAMA_HOST: http://ollama:11434
    entrypoint:
    - sh
    - -c
    - ollama show llama3.1:8b >/dev/null 2>&1 || ollama pull llama3.1:8b
    depends_on:
      ollama:
        condition: service_healthy
    restart: 'no'
  chroma:
    image: chromadb/chroma:${CHROMA_TAG:?pin CHROMA_TAG in .env}
    volumes:
    - chroma_data:/data
    healthcheck:
      test:
      - CMD-SHELL
      - bash -c ':> /dev/tcp/127.0.0.1/8000' || exit 1
      interval: 10s
      timeout: 5s
      retries: 12
    restart: unless-stopped
  api:
    build:
      context: .
      doc

**পড়ার নিয়ম:**

* প্রতিটা healthcheck সেই image-এ যা আছে তা দিয়েই লেখা: `ollama/ollama`-এ `curl` নেই → `ollama list` (CLI server-কে ডাকে, server না থাকলে fail)। `chromadb/chroma` debian-slim, `curl` নেই কিন্তু `bash` আছে → `/dev/tcp` দিয়ে TCP check (ceiling: port খোলা মানে API পুরো ready এমন না)। আমাদের নিজের Python image-এ `urllib`।
* Healthcheck-এর ভেতরে `localhost` ঠিক আছে — check container-এর ভেতরেই চলে, নিজেকেই দেখে। ভুল হতো `environment`-এর URL-এ।
* `api`-র `start_period` 120 s — embedding model load আর `HC_AUTO_INGEST`-এর ingest শেষ হওয়ার সময়। এটা ছোট হলে ধীর machine-এ API `unhealthy`, আর UI কখনো start হয় না।

## 2. `docker compose config` — Compose নিজে কী ধরে

`docker compose config` file পড়ে, `.env` থেকে variable বসায়, reference মেলায় (volume declared কিনা, `depends_on`-এর service আছে কিনা), আর resolved file দেখায়। Daemon লাগে না — CI-র প্রথম check হিসেবে চমৎকার। তিনটা case: ঠিক file + `.env`, `.env` ছাড়া, আর একটা undeclared volume। `.env`-এর `x.y.z` placeholder — Docker Hub-এ যে version নিজে test করেছেন সেটা বসান।

In [3]:
ENV_EXAMPLE = "OLLAMA_TAG=x.y.z\nCHROMA_TAG=x.y.z\nHC_API_KEY=\n"

def compose_config(spec, env_text, *args):
    d = Path(tempfile.mkdtemp(dir=WORK))
    (d / "compose.yaml").write_text(to_yaml(spec), encoding="utf-8")
    if env_text is not None:
        (d / ".env").write_text(env_text, encoding="utf-8")
    r = subprocess.run([DOCKER, "compose", "config", *args], cwd=d, capture_output=True, text=True)
    return r.returncode, (r.stdout or r.stderr).strip().replace("\n", " | ")

if DOCKER:
    undeclared = copy.deepcopy(STACK)
    undeclared["services"]["chroma"]["volumes"] = ["chroma_db:/data"]
    for label, spec, env in [("good file + .env", STACK, ENV_EXAMPLE),
                             ("good file, no .env", STACK, None),
                             ("undeclared volume", undeclared, ENV_EXAMPLE)]:
        rc, out = compose_config(spec, env, "--services")
        print(f"[{label}] exit={rc}: {out[:160]}")
else:
    print("docker CLI not found - skipped. On your machine: docker compose config --quiet")

[good file + .env] exit=0: ollama | ollama-init | chroma | api | ui


[good file, no .env] exit=1: error while interpolating services.ollama.image: required variable OLLAMA_TAG is missing a value: pin OLLAMA_TAG in .env


[undeclared volume] exit=1: service "chroma" refers to undefined volume chroma_db: invalid compose project


**পড়ার নিয়ম:** ঠিক file-এ Compose service-এর তালিকা দেয়। `.env` না থাকলে `${OLLAMA_TAG:?}`-এর কারণে থামে — tag pin করতে বাধ্য করে। Undeclared volume-ও ধরে। কিন্তু Compose শুধু file-টা *বৈধ* কিনা দেখে, *ঠিক* কিনা দেখে না। পরের section-এর ভুলগুলোর একটাও সে ধরে না।

## 3. Compose Linter — Compose যা ধরে না

নিজের linter, ~৪০ লাইন, এই stack-এর সবচেয়ে সাধারণ ভুলগুলোর জন্য:

| Rule | কেন |
|---|---|
| `localhost` env URL-এ | container-এর `localhost` সে নিজে |
| healthcheck ছাড়া `service_healthy` | dependency কখনো healthy হবে না → `up` ব্যর্থ |
| list-form `depends_on` | শুধু "started" পর্যন্ত অপেক্ষা |
| `latest` / tag নেই | আজ আর কাল আলাদা stack |
| literal secret | file commit হলেই ফাঁস |
| stateful service-এ volume নেই | recreate-এ model/vector হারায় |
| Ollama/Chroma port public | দুটোরই auth নেই |
| undeclared named volume | Compose-ও ধরে |

In [4]:
SECRET_NAME = re.compile(r"KEY|SECRET|PASSWORD|TOKEN", re.I)
STATEFUL = ("ollama/ollama", "chromadb/chroma")

def lint_compose(spec):
    services, declared, out = spec.get("services", {}), set(spec.get("volumes") or {}), []
    add = lambda sev, svc, rule, msg: out.append({"severity": sev, "service": svc, "rule": rule, "message": msg})
    for name, s in services.items():
        image, one_shot, vols = s.get("image", ""), s.get("restart") == "no", s.get("volumes", [])
        deps = s.get("depends_on", {})
        if isinstance(deps, list):
            add("WARN", name, "depends_on-list", f"{deps} only waits for 'started'; use condition: service_healthy")
            deps = {}
        for dep, cond in deps.items():
            if cond.get("condition") == "service_healthy" and "healthcheck" not in services.get(dep, {}):
                add("ERROR", name, "healthy-needs-healthcheck", f"waits for {dep} to be healthy, but {dep} has no healthcheck")
        env = s.get("environment", {})
        pairs = env.items() if isinstance(env, dict) else (e.split("=", 1) for e in env)
        for k, v in pairs:
            v = str(v)
            if re.search(r"\b(localhost|127\.0\.0\.1)\b", v):
                add("ERROR", name, "localhost-between-containers", f"{k}={v}: localhost is this container; use the service name")
            if SECRET_NAME.search(k) and v and not v.startswith("${"):
                add("ERROR", name, "secret-literal", f"{k} has a literal value; use ${{{k}}} from .env")
        if image and image.rsplit("/", 1)[-1].partition(":")[2] in ("", "latest"):
            add("WARN", name, "unpinned-tag", f"{image}: pin the version you tested")
        for v in vols:
            src = v.split(":", 1)[0]
            if not src.startswith((".", "/", "~", "$")) and src not in declared:
                add("ERROR", name, "undeclared-volume", f"{src} is not under top-level volumes:")
        if image.startswith(STATEFUL) and not one_shot and not vols:
            add("ERROR", name, "stateful-without-volume", "models/vectors live in the container layer -> lost on recreate")
        if image.startswith(STATEFUL):
            for p in s.get("ports", []):
                if not str(p).startswith("127.0.0.1:"):
                    add("WARN", name, "internal-port-public", f"{p}: published on all interfaces, no auth")
        if not one_shot and "healthcheck" not in s:
            add("INFO", name, "no-healthcheck", "nobody can wait for this service to be ready")
    return pd.DataFrame(out, columns=["severity", "service", "rule", "message"])

assert lint_compose(STACK).empty, lint_compose(STACK)        # our stack is clean
print("STACK: 0 findings")

STACK: 0 findings


In [5]:
BAD = copy.deepcopy(STACK)
b = BAD["services"]
b["ollama"].update(image="ollama/ollama:latest", ports=["11434:11434"]); del b["ollama"]["volumes"]
del b["chroma"]["healthcheck"]
b["api"]["environment"].update(HC_OLLAMA_HOST="http://localhost:11434", HC_API_KEY="sk-demo-123")
b["ui"]["depends_on"] = ["api"]

findings = lint_compose(BAD)
assert set(findings.rule) >= {"localhost-between-containers", "healthy-needs-healthcheck", "secret-literal",
                              "unpinned-tag", "stateful-without-volume", "internal-port-public", "depends_on-list"}
if DOCKER:
    rc, _ = compose_config(BAD, ENV_EXAMPLE, "--quiet")
    print(f"docker compose config on BAD: exit={rc} (0 = 'valid')")
findings

docker compose config on BAD: exit=0 (0 = 'valid')


,severity,service,rule,message
0,WARN,ollama,unpinned-tag,ollama/ollama:latest: pin the version you tested
1,ERROR,ollama,stateful-without-volume,models/vectors live in the container layer -> ...
2,WARN,ollama,internal-port-public,"11434:11434: published on all interfaces, no auth"
3,INFO,chroma,no-healthcheck,nobody can wait for this service to be ready
4,ERROR,api,healthy-needs-healthcheck,"waits for chroma to be healthy, but chroma has..."
5,ERROR,api,localhost-between-containers,HC_OLLAMA_HOST=http://localhost:11434: localho...
6,ERROR,api,secret-literal,HC_API_KEY has a literal value; use ${HC_API_K...
7,WARN,ui,depends_on-list,['api'] only waits for 'started'; use conditio...


**পড়ার নিয়ম:** Compose বলে BAD file "valid" (exit 0), অথচ এতে চারটা ERROR — প্রতিটা `docker compose up`-এর পরে কোনো না কোনোভাবে ভাঙত: API Ollama খুঁজে পেত না (`localhost`), `up` chroma-র health-এর জন্য আটকে যেত, secret repo-তে যেত, আর প্রথম recreate-এ ৪.৯ GB model হারাত। Linter-টা CI-তে `docker compose config`-এর পরের ধাপ হিসেবে বসানো যায়।

## 4. Dockerfile Linter

একই চিন্তা Dockerfile-এ। একটা "naive" Dockerfile (laptop থেকে সরাসরি কপি করার মতো) আর lecture-এর ভালো Dockerfile — দুটোই একই linter-এ।

In [6]:
NAIVE = """
FROM python:latest
ENV HC_API_KEY=sk-demo-123
WORKDIR /app
COPY . .
RUN pip install torch sentence-transformers fastapi uvicorn chromadb
CMD uvicorn app.main:app --host 0.0.0.0 --port 8000
"""

GOOD = """
FROM python:3.12-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1 HF_HOME=/home/appuser/.cache/huggingface
RUN useradd --create-home appuser
WORKDIR /app
COPY requirements.txt .
RUN pip install --index-url https://download.pytorch.org/whl/cpu torch && pip install -r requirements.txt
USER appuser
RUN python -c "from sentence_transformers import SentenceTransformer as S; S('sentence-transformers/paraphrase-MiniLM-L3-v2')"
ENV HF_HUB_OFFLINE=1
COPY --chown=appuser app ./app
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s --start-period=120s CMD python -c "import urllib.request as u; u.urlopen('http://localhost:8000/health')"
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

def lint_dockerfile(text):
    lines = [l.strip() for l in text.splitlines() if l.strip() and not l.strip().startswith("#")]
    out = []
    add = lambda sev, rule, msg: out.append({"severity": sev, "rule": rule, "message": msg})
    first = lambda pred: next((i for i, l in enumerate(lines) if pred(l)), None)
    base = next((l.split()[1] for l in lines if l.upper().startswith("FROM ")), "")
    if base.rsplit("/", 1)[-1].partition(":")[2] in ("", "latest"):
        add("WARN", "unpinned-base", f"FROM {base}: pin a tag (and prefer -slim)")
    copy_all, pip = first(lambda l: re.match(r"COPY\s+\.\s", l, re.I)), first(lambda l: "pip install" in l)
    if copy_all is not None and pip is not None and copy_all < pip:
        add("WARN", "cache-bust", "COPY . . before pip install: every code edit reinstalls torch")
    if any("pip install" in l and re.search(r"\btorch\b", l) for l in lines) and "download.pytorch.org/whl/cpu" not in text:
        add("WARN", "cuda-torch", "default torch wheel pulls GBs of CUDA libs; the LLM runs in Ollama -> CPU index")
    if not any(l.upper().startswith("USER ") for l in lines):
        add("WARN", "root-user", "container runs as root")
    if not any(l.upper().startswith("HEALTHCHECK ") for l in lines):
        add("INFO", "no-healthcheck", "fine if compose defines one")
    for l in lines:
        if re.match(r"(ENV|ARG)\s", l, re.I) and any(SECRET_NAME.search(t.split("=")[0]) for t in l.split()[1:]):
            add("ERROR", "secret-in-image", f"{l} -> visible in docker history / image inspect")
        if re.match(r"RUN\s.*ollama pull", l, re.I):
            add("ERROR", "llm-baked", "LLM weights in the image: GBs per push/pull; use a volume + init pull")
    cmd = next((l for l in reversed(lines) if l.upper().startswith(("CMD ", "ENTRYPOINT "))), "")
    if cmd and not cmd.split(None, 1)[1].startswith("["):
        add("WARN", "shell-form-cmd", "PID 1 is /bin/sh: SIGTERM never reaches uvicorn")
    return pd.DataFrame(out, columns=["severity", "rule", "message"])

assert lint_dockerfile(GOOD).empty, lint_dockerfile(GOOD)
print("GOOD Dockerfile: 0 findings")
lint_dockerfile(NAIVE)

GOOD Dockerfile: 0 findings


,severity,rule,message
0,WARN,unpinned-base,FROM python:latest: pin a tag (and prefer -slim)
1,WARN,cache-bust,COPY . . before pip install: every code edit r...
2,WARN,cuda-torch,default torch wheel pulls GBs of CUDA libs; th...
3,WARN,root-user,container runs as root
4,INFO,no-healthcheck,fine if compose defines one
5,ERROR,secret-in-image,ENV HC_API_KEY=sk-demo-123 -> visible in docke...
6,WARN,shell-form-cmd,PID 1 is /bin/sh: SIGTERM never reaches uvicorn


**পড়ার নিয়ম:** Naive file-এর প্রতিটা লাইনে একটা দাম: `python:latest` (≈১ GB full image, version চলমান), image-এ secret, `COPY . .` আগে (code-এর প্রতিটা edit-এ torch আবার install), default torch (CUDA, কয়েক GB), root, shell-form `CMD` (`docker stop` ১০ s অপেক্ষা করে kill)। GOOD file-এ শূন্য — আর লক্ষ্য করুন সেখানে model bake `COPY app`-এর **আগে**, আর `HF_HUB_OFFLINE=1` bake-এর **পরে**।

এটা একটা শেখার linter, সম্পূর্ণ না — production-এ hadolint-এর মতো tool আর image scanner যোগ করুন।

## 5. Image Size Budget

সংখ্যাগুলো **approximate** installed size (MB, Linux x86_64) — package version আর platform-ভেদে বদলায়। শেখার বিষয় পদ্ধতি: প্রতিটা অংশের একটা budget, আর কোন সিদ্ধান্ত কতটা বাঁচায়। নিজের image `docker image ls` আর `docker history <image>` দিয়ে মেপে এই table-এ বসান।

In [7]:
# (naive, lean) approximate MB. Replace with your own `docker history` numbers.
BUDGET = {
    "base image (python:3.12 vs 3.12-slim)": (1000, 125),
    "torch (default CUDA wheel vs CPU-only wheel)": (4500, 700),
    "transformers + sentence-transformers + numpy/scipy": (250, 250),
    "vector DB client (chromadb vs chromadb-client)": (200, 30),
    "embedding model paraphrase-MiniLM-L3-v2 (baked)": (61, 61),
    "LLM llama3.1:8b Q4_K_M (baked vs Ollama volume)": (4900, 0),
    "pip cache left in a layer": (700, 0),
}
budget = pd.DataFrame(BUDGET, index=["naive_MB", "lean_MB"]).T
budget.loc["TOTAL"] = budget.sum()
budget["saved_MB"] = budget.naive_MB - budget.lean_MB
naive, lean = budget.loc["TOTAL", ["naive_MB", "lean_MB"]]
print(f"API image: naive ~{naive/1000:.1f} GB vs lean ~{lean/1000:.1f} GB -> {naive/lean:.0f}x smaller")
for up_MBps in (2, 10):
    print(f"push at {up_MBps} MB/s: naive {naive/up_MBps/60:.0f} min, lean {lean/up_MBps/60:.0f} min")
budget

API image: naive ~11.6 GB vs lean ~1.2 GB -> 10x smaller
push at 2 MB/s: naive 97 min, lean 10 min
push at 10 MB/s: naive 19 min, lean 2 min


,naive_MB,lean_MB,saved_MB
base image (python:3.12 vs 3.12-slim),1000,125,875
torch (default CUDA wheel vs CPU-only wheel),4500,700,3800
transformers + sentence-transformers + numpy/scipy,250,250,0
vector DB client (chromadb vs chromadb-client),200,30,170
embedding model paraphrase-MiniLM-L3-v2 (baked),61,61,0
LLM llama3.1:8b Q4_K_M (baked vs Ollama volume),4900,0,4900
pip cache left in a layer,700,0,700
TOTAL,11611,1166,10445


In [8]:
# The model is downloaded once by ollama-init. How long is "once"? (example bandwidths)
MODEL_GB = 4.9
pd.DataFrame([{"bandwidth_MB_per_s": bw, "first_up_minutes": round(MODEL_GB * 1000 / bw / 60, 1),
               "second_up_with_volume_s": "<1 (ollama show finds it)"} for bw in (2, 5, 20, 50)])

,bandwidth_MB_per_s,first_up_minutes,second_up_with_volume_s
0,2,40.8,<1 (ollama show finds it)
1,5,16.3,<1 (ollama show finds it)
2,20,4.1,<1 (ollama show finds it)
3,50,1.6,<1 (ollama show finds it)


**পড়ার নিয়ম:** Image-এর ৮০%-এর বেশি দুটো সিদ্ধান্তে: CUDA torch আর baked LLM। দুটোই এই container-এ অপ্রয়োজনীয় — GPU কাজ Ollama container করে, আর LLM থাকে volume-এ। প্রথম `up`-এ ৪.৯ GB download এড়ানো যায় না (২ MB/s-এ ৪০ মিনিটের বেশি — README-তে এটা বলে রাখুন), কিন্তু volume থাকলে দ্বিতীয়বার থেকে এক সেকেন্ডও লাগে না।

## 6. `docker compose ps` পড়া — Running বনাম Ready

`docker compose ps -a --format json` প্রতিটা container-এর `State` (`running`, `exited`, `created`...) আর `Health` (`starting`, `healthy`, `unhealthy`, বা healthcheck না থাকলে খালি) দেয়। `-a` না দিলে exit করা `ollama-init` দেখাই যায় না। নতুন Compose প্রতি লাইনে একটা JSON object দেয়, পুরোনোগুলো একটা JSON array — parser দুটোই নেয়।

⚠️ নিচের তিনটা snapshot **illustrative** — চলমান stack থেকে না (এই machine-এ daemon ছিল না), field-গুলো Compose-এর output-এর আকারে হাতে লেখা।

In [9]:
SAMPLE_PS = {
    "first start, model downloading": """
{"Service":"ollama","State":"running","Health":"healthy","ExitCode":0,"Status":"Up 2 minutes (healthy)"}
{"Service":"ollama-init","State":"running","Health":"","ExitCode":0,"Status":"Up 2 minutes"}
{"Service":"chroma","State":"running","Health":"healthy","ExitCode":0,"Status":"Up 2 minutes (healthy)"}
{"Service":"api","State":"created","Health":"","ExitCode":0,"Status":"Created"}
{"Service":"ui","State":"created","Health":"","ExitCode":0,"Status":"Created"}""",
    "ready": """[
{"Service":"ollama","State":"running","Health":"healthy","ExitCode":0,"Status":"Up 6 minutes (healthy)"},
{"Service":"ollama-init","State":"exited","Health":"","ExitCode":0,"Status":"Exited (0) 2 minutes ago"},
{"Service":"chroma","State":"running","Health":"healthy","ExitCode":0,"Status":"Up 6 minutes (healthy)"},
{"Service":"api","State":"running","Health":"healthy","ExitCode":0,"Status":"Up 2 minutes (healthy)"},
{"Service":"ui","State":"running","Health":"healthy","ExitCode":0,"Status":"Up 1 minute (healthy)"}]""",
    "offline, init used plain pull": """
{"Service":"ollama","State":"running","Health":"healthy","ExitCode":0,"Status":"Up 1 minute (healthy)"}
{"Service":"ollama-init","State":"exited","Health":"","ExitCode":1,"Status":"Exited (1) 40 seconds ago"}
{"Service":"chroma","State":"running","Health":"healthy","ExitCode":0,"Status":"Up 1 minute (healthy)"}
{"Service":"api","State":"created","Health":"","ExitCode":0,"Status":"Created"}
{"Service":"ui","State":"created","Health":"","ExitCode":0,"Status":"Created"}""",
}
ONE_SHOT = {"ollama-init"}

def parse_ps(text):
    text = text.strip()
    return json.loads(text) if text.startswith("[") else [json.loads(l) for l in text.splitlines() if l.strip()]

def verdict(row):
    state, health, code = row["State"], row.get("Health", ""), row.get("ExitCode", 0)
    if row["Service"] in ONE_SHOT:                       # a one-shot is DONE when it exited 0
        return "OK" if state == "exited" and code == 0 else "FAIL" if state == "exited" else "WAIT"
    if state in ("exited", "dead", "restarting") or health == "unhealthy":
        return "FAIL"
    return "OK" if state == "running" and health in ("healthy", "") else "WAIT"

def stack_status(text):
    rows = [{"service": r["Service"], "status": r["Status"], "verdict": verdict(r)} for r in parse_ps(text)]
    v = {r["verdict"] for r in rows}
    overall = "FAIL" if "FAIL" in v else "WAIT" if "WAIT" in v else "READY"
    hint = next((f"docker compose logs {r['service']}" for r in rows if r["verdict"] == "FAIL"), "")
    return overall, hint, pd.DataFrame(rows)

for label, text in SAMPLE_PS.items():
    overall, hint, table = stack_status(text)
    print(f"[{label}] -> {overall} {('| next: ' + hint) if hint else ''}")
assert [stack_status(t)[0] for t in SAMPLE_PS.values()] == ["WAIT", "READY", "FAIL"]
stack_status(SAMPLE_PS["offline, init used plain pull"])[2]

[first start, model downloading] -> WAIT 
[ready] -> READY 
[offline, init used plain pull] -> FAIL | next: docker compose logs ollama-init


,service,status,verdict
0,ollama,Up 1 minute (healthy),OK
1,ollama-init,Exited (1) 40 seconds ago,FAIL
2,chroma,Up 1 minute (healthy),OK
3,api,Created,WAIT
4,ui,Created,WAIT


**পড়ার নিয়ম:**

* প্রথম snapshot-এ `api` আর `ui` "Created" — ভুল না; তারা `ollama-init` শেষ হওয়ার অপেক্ষায় (`service_completed_successfully`)। Model download চলছে।
* `ready`-তে `ollama-init` "Exited (0)" — এটা সাফল্য, error না। একটা সাধারণ readiness script "সব container running?" জিজ্ঞেস করলে এখানে ভুল করে FAIL বলত।
* তৃতীয়টা Class-এর সবচেয়ে শিক্ষণীয় failure: model volume-এ আছে, কিন্তু internet নেই আর init-এ শুধু `ollama pull` → exit 1 → API কখনো start হয় না। `ollama show ... || ollama pull ...` এই ঘটনাটাই আটকায়।

## 7. Single-Click-এর শেষ ধাপ: Readiness Wait

`docker compose up -d` dependency-র condition-এর জন্য নিজেই অপেক্ষা করে, কিন্তু শেষ service (UI) নিজে ready কিনা সেটা দেখে না। Single-click script তাই একটা loop দিয়ে শেষ করে: health URL-এ 200 না আসা পর্যন্ত poll, timeout-এ পরিষ্কার message। নিচে একটা নকল server (প্রথম ৩ বার 503, তারপর 200) দিয়ে সেই loop — Docker লাগে না।

In [10]:
class FlakyHealth(BaseHTTPRequestHandler):
    calls = 0
    def do_GET(self):
        FlakyHealth.calls += 1
        self.send_response(200 if FlakyHealth.calls > 3 else 503)
        self.end_headers()
    def log_message(self, *args):
        pass

def wait_until_ready(url, timeout_s=600, every_s=5):
    t0, tries = time.monotonic(), 0
    while time.monotonic() - t0 < timeout_s:
        tries += 1
        try:
            with urllib.request.urlopen(url, timeout=3) as r:
                if r.status == 200:
                    return {"ready": True, "tries": tries, "seconds": round(time.monotonic() - t0, 2)}
        except (urllib.error.URLError, OSError):     # 503 (HTTPError) and "connection refused" both mean: not yet
            pass
        time.sleep(every_s)
    return {"ready": False, "tries": tries, "seconds": round(time.monotonic() - t0, 2)}

srv = ThreadingHTTPServer(("127.0.0.1", 0), FlakyHealth)
threading.Thread(target=srv.serve_forever, daemon=True).start()
flaky = wait_until_ready(f"http://127.0.0.1:{srv.server_port}/_stcore/health", timeout_s=5, every_s=0.2)
srv.shutdown()
nothing = wait_until_ready("http://127.0.0.1:9/_stcore/health", timeout_s=1, every_s=0.2)
assert flaky["ready"] and flaky["tries"] == 4 and not nothing["ready"]
pd.DataFrame([{"case": "UI starting (503 x3, then 200)"} | flaky, {"case": "nothing listening"} | nothing])

,case,ready,tries,seconds
0,"UI starting (503 x3, then 200)",True,4,0.72
1,nothing listening,False,1,2.23


**পড়ার নিয়ম:** ৪র্থ চেষ্টায় ready — script তখনই "Ready: http://localhost:8501" বলে। কেউ না শুনলে timeout-এ থামে, চিরকাল ঝুলে থাকে না। আসল `start.sh`-এ একই loop `curl -fs`-এ, `start.ps1`-এ `Invoke-WebRequest`-এ, timeout ১০ মিনিট (প্রথম model download-এর জন্য)। Timeout হলে script `docker compose ps -a` দেখিয়ে বলে কোন log দেখতে হবে — section 6-এর `stack_status`-এর কাজটাই।

নিজের machine-এ পুরো flow (illustrative — এই machine-এ চালানো হয়নি):

```text
$ ./start.sh
 ✔ Container healthcare-assistant-ollama-1        Healthy
 ✔ Container healthcare-assistant-chroma-1        Healthy
 ✔ Container healthcare-assistant-ollama-init-1   Exited
 ✔ Container healthcare-assistant-api-1           Healthy
 ✔ Container healthcare-assistant-ui-1            Started
Ready: http://localhost:8501
```

## 📝 Key Takeaways

* **Container ক্ষণস্থায়ী, volume টেকসই।** Model আর vector DB volume-এ; top-level `name:` volume-এর নাম স্থির রাখে; `down -v` data মোছে।
* **Container-এর `localhost` সে নিজে।** Service name দিয়ে ডাকুন (`http://ollama:11434`); healthcheck-এর ভেতরে `localhost` ঠিক।
* **Running ≠ ready।** Healthcheck (image-এ যে tool আছে তা দিয়ে) + `condition: service_healthy`; one-shot-এ `service_completed_successfully`।
* **Init idempotent হতে হবে।** `ollama show || ollama pull` — model থাকলে network ছোঁয় না, offline-এও stack ওঠে।
* **Image budget:** CUDA torch আর baked LLM-ই ৮০%+; CPU wheel + volume-এ ≈১০ গুণ ছোট। Layer-এর ক্রম: requirements → model → code।
* **`docker compose config` বৈধতা দেখে, সঠিকতা না** — নিজের lint যোগ করুন। Single-click = এক command + idempotent + readiness wait + পরিষ্কার failure।

## 🧠 Brain Teasers

1. Section 3-এর BAD stack-এ `ollama`-কে `ports: ["127.0.0.1:11434:11434"]` দিন। কোন finding চলে যায়? কেন এটা এখনো আদর্শ না?
2. `lint_compose`-এ একটা নতুন rule যোগ করুন: যে service-এর `ports` আছে কিন্তু `restart` policy নেই। BAD stack-এ কয়টা ধরা পড়ে?
3. Section 5-এ `all-mpnet-base-v2` (≈৪২০ MB) বসান। Lean image কত বাড়ে? ২ MB/s-এ push কত মিনিট বাড়ে?
4. Section 6-এ একটা snapshot বানান যেখানে `api` `"Health":"unhealthy"`। `stack_status` কী বলে? Docker কি নিজে থেকে API restart করবে?
5. Section 7-এ `every_s=5` আর `timeout_s=600` আসল মান। Model download ২০ মিনিট নিলে কী হয়? Timeout কীভাবে বাছবেন — আর script-কে কীভাবে বলবেন "download চলছে, error না"?